# ML Walk-Forward Factor Backtest Definitive

Cross-sectional model using engineered alpha factors and a simple long/short portfolio.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

pd.set_option('display.max_columns', 80)
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == '04_alpha_factor_research' else Path.cwd().resolve()
if str(REPO_ROOT / '04_alpha_factor_research') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / '04_alpha_factor_research'))
from alpha_factor_utils import load_market_panel, compute_technical_factors, walk_forward_splits, long_short_returns, performance_summary
OUTPUT = REPO_ROOT / 'data' / 'alpha_factor_research'
OUTPUT.mkdir(parents=True, exist_ok=True)
prices = load_market_panel(REPO_ROOT)
panel = compute_technical_factors(prices)
factors = [c for c in panel.columns if c.endswith('_rank')]
target = 'return_5d'
model_data = panel.dropna(subset=factors + [target]).copy()
splits = walk_forward_splits(model_data['date'], train_days=252, test_days=63)
len(model_data), len(factors), len(splits)

(2615, 6, 2)

In [2]:
predictions = []
metrics = []
for i, (train_start, train_end, test_start, test_end) in enumerate(splits, 1):
    train = model_data[(model_data['date'] >= train_start) & (model_data['date'] <= train_end)]
    test = model_data[(model_data['date'] >= test_start) & (model_data['date'] <= test_end)]
    if len(train) < 100 or test.empty:
        continue
    model = HistGradientBoostingRegressor(max_iter=80, learning_rate=0.05, max_leaf_nodes=15, random_state=i)
    try:
        model.fit(train[factors], train[target])
    except Exception:
        model = RandomForestRegressor(n_estimators=80, min_samples_leaf=10, random_state=i, n_jobs=-1)
        model.fit(train[factors], train[target])
    pred = test[['date','ticker',target] + factors].copy()
    pred['prediction'] = model.predict(test[factors])
    pred['split'] = i
    predictions.append(pred)
    metrics.append({'split': i, 'train_start': train_start, 'train_end': train_end, 'test_start': test_start, 'test_end': test_end, 'rmse': float(np.sqrt(np.mean((test[target].to_numpy() - pred['prediction'].to_numpy()) ** 2)))})

predictions = pd.concat(predictions, ignore_index=True) if predictions else model_data[['date','ticker',target] + factors].assign(prediction=0.0, split=0)
metrics = pd.DataFrame(metrics)
ls = long_short_returns(predictions, score_col='prediction', return_col=target)
summary = performance_summary(ls['long_short_return'] if not ls.empty else pd.Series(dtype=float))
predictions.to_parquet(OUTPUT / 'walk_forward_ml_predictions.parquet', index=False)
metrics.to_csv(OUTPUT / 'walk_forward_ml_metrics.csv', index=False)
ls.to_parquet(OUTPUT / 'walk_forward_ml_long_short_returns.parquet', index=False)
display(metrics.tail())
display(ls.tail())
summary

,split,train_start,train_end,test_start,test_end,rmse
0,1,2022-03-31,2023-03-17,2023-03-20,2023-06-14,0.035856
1,2,2022-06-28,2023-06-14,2023-06-15,2023-09-11,0.035741


,date,long_return,short_return,long_short_return
121,2023-09-05,-0.006918,0.018759,-0.025677
122,2023-09-06,0.010496,-0.017292,0.027787
123,2023-09-07,0.022743,-0.011090,0.033833
124,2023-09-08,0.033790,-0.016481,0.050271
125,2023-09-11,0.019859,-0.015111,0.034970


BacktestSummary(annual_return=0.30996648917190206, annual_volatility=0.5487914223633135, sharpe=0.5648165706327251, max_drawdown=-0.5400895447793742, hit_rate=0.4603174603174603)